In [1]:
import pandas as pd

# 1. Cargamos el dataset sucio
df = pd.read_csv('laptopData sucio .csv')

# 2. Visualizamos las primeras 5 filas para ver el "caos"
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1.0,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2.0,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3.0,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


In [7]:
# 1. Aseguramos que la columna sea de tipo string y quitamos 'GB'
df['Ram'] = df['Ram'].astype(str).str.replace('GB', '', regex=False)

# 2. Convertimos a numérico, convirtiendo errores (como valores vacíos o 'nan') en NaN
df['Ram'] = pd.to_numeric(df['Ram'], errors='coerce')

# 3. Rellenamos los NaN con 0 y convertimos a entero
df['Ram'] = df['Ram'].fillna(0).astype(int)

# 4. Comprobamos el cambio
print(f'Nuevo tipo de dato: {df['Ram'].dtype}')
df[['Company', 'Ram']].head()

Nuevo tipo de dato: int64


,Company,Ram
0,Apple,8
1,Apple,8
2,HP,8
3,Apple,16
4,Apple,8


In [8]:
# 1. Quitamos las letras 'kg'
df['Weight'] = df['Weight'].str.replace('kg', '', regex=False)

# 2. Convertimos a float usando pd.to_numeric
# errors='coerce' transformará valores no numéricos (como '?') en NaN automáticamente
df['Weight'] = pd.to_numeric(df['Weight'], errors='coerce')

# 3. Comprobamos
print(f'Nuevo tipo de dato: {df['Weight'].dtype}')
print(f'Valores nulos encontrados: {df['Weight'].isna().sum()}')
df[['Company', 'Weight']].head()

Nuevo tipo de dato: float64
Valores nulos encontrados: 31


,Company,Weight
0,Apple,1.37
1,Apple,1.34
2,HP,1.86
3,Apple,1.83
4,Apple,1.37


In [9]:
# 1. Redondeamos el precio a 0 decimales (para quitar los centavos largos)
df['Price'] = df['Price'].round(0)

# 2. Le cambiamos el nombre a la columna para aplicar las buenas prácticas
df = df.rename(columns={'Price': 'Price_INR'})

# 3. Vemos cómo quedó la tabla hasta ahora
df[['Company', 'Ram', 'Weight', 'Price_INR']].head()

,Company,Ram,Weight,Price_INR
0,Apple,8,1.37,71379.0
1,Apple,8,1.34,47896.0
2,HP,8,1.86,30636.0
3,Apple,16,1.83,135195.0
4,Apple,8,1.37,96096.0


In [10]:
# 1. Separamos la columna Memory usando el '+' como punto de corte.
# El parámetro 'expand=True' obliga a Python a crear columnas separadas.
discos = df['Memory'].str.split('+', n=1, expand=True)

# 2. Bautizamos y guardamos las nuevas columnas en nuestro DataFrame
df['Disco_1'] = discos[0]
df['Disco_2'] = discos[1]

# 3. Miramos cómo quedó (traemos las primeras 10 filas para ver variedad)
df[['Memory', 'Disco_1', 'Disco_2']].head(10)

,Memory,Disco_1,Disco_2
0,128GB SSD,128GB SSD,None
1,128GB Flash Storage,128GB Flash Storage,None
2,256GB SSD,256GB SSD,None
3,512GB SSD,512GB SSD,None
4,256GB SSD,256GB SSD,None
5,500GB HDD,500GB HDD,None
6,256GB Flash Storage,256GB Flash Storage,None
7,256GB Flash Storage,256GB Flash Storage,None
8,512GB SSD,512GB SSD,None
9,256GB SSD,256GB SSD,None


In [11]:
# 1. Extraemos el número base del Disco_1 y lo convertimos a float
df['Cap_Disco_1_GB'] = df['Disco_1'].str.extract('(\d+)').astype(float)

# 2. Magia de Pandas: Localizamos las filas que dicen "TB" y multiplicamos su capacidad por 1000
df.loc[df['Disco_1'].str.contains('TB', na=False), 'Cap_Disco_1_GB'] = df['Cap_Disco_1_GB'] * 1000

# 3. Comprobamos la cirugía
df[['Disco_1', 'Cap_Disco_1_GB']].head(10)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_10143/247255154.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['Cap_Disco_1_GB'] = df['Disco_1'].str.extract('(\d+)').astype(float)


,Disco_1,Cap_Disco_1_GB
0,128GB SSD,128.0
1,128GB Flash Storage,128.0
2,256GB SSD,256.0
3,512GB SSD,512.0
4,256GB SSD,256.0
5,500GB HDD,500.0
6,256GB Flash Storage,256.0
7,256GB Flash Storage,256.0
8,512GB SSD,512.0
9,256GB SSD,256.0


In [12]:
# 1. Extraemos el número base del Disco_1 y lo convertimos a float
df['Cap_Disco_2_GB'] = df['Disco_1'].str.extract('(\d+)').astype(float)

# 2. Magia de Pandas: Localizamos las filas que dicen "TB" y multiplicamos su capacidad por 1000
df.loc[df['Disco_2'].str.contains('TB', na=False), 'Cap_Disco_2_GB'] = df['Cap_Disco_2_GB'] * 1000

# 3. Comprobamos la cirugía
df[['Disco_1', 'Cap_Disco_2_GB']].head(10)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_10143/3171301979.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['Cap_Disco_2_GB'] = df['Disco_1'].str.extract('(\d+)').astype(float)


,Disco_1,Cap_Disco_2_GB
0,128GB SSD,128.0
1,128GB Flash Storage,128.0
2,256GB SSD,256.0
3,512GB SSD,512.0
4,256GB SSD,256.0
5,500GB HDD,500.0
6,256GB Flash Storage,256.0
7,256GB Flash Storage,256.0
8,512GB SSD,512.0
9,256GB SSD,256.0


In [13]:
# 1. Sumamos las columnas rellenando los nulos con 0 solo para esta operación
df['Capacidad_Total_GB'] = df['Cap_Disco_1_GB'].fillna(0) + df['Cap_Disco_2_GB'].fillna(0)

# 2. Vemos el resultado final de nuestra obra de arte
df[['Memory', 'Cap_Disco_1_GB', 'Cap_Disco_2_GB', 'Capacidad_Total_GB']].head(10)

,Memory,Cap_Disco_1_GB,Cap_Disco_2_GB,Capacidad_Total_GB
0,128GB SSD,128.0,128.0,256.0
1,128GB Flash Storage,128.0,128.0,256.0
2,256GB SSD,256.0,256.0,512.0
3,512GB SSD,512.0,512.0,1024.0
4,256GB SSD,256.0,256.0,512.0
5,500GB HDD,500.0,500.0,1000.0
6,256GB Flash Storage,256.0,256.0,512.0
7,256GB Flash Storage,256.0,256.0,512.0
8,512GB SSD,512.0,512.0,1024.0
9,256GB SSD,256.0,256.0,512.0


In [14]:
# Muestra las primeras 10 filas de TODAS las columnas
df.head(10)

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_INR,Disco_1,Disco_2,Cap_Disco_1_GB,Cap_Disco_2_GB,Capacidad_Total_GB
0,0.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71379.0,128GB SSD,None,128.0,128.0,256.0
1,1.0,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47896.0,128GB Flash Storage,None,128.0,128.0,256.0
2,2.0,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0,256GB SSD,None,256.0,256.0,512.0
3,3.0,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.0,512GB SSD,None,512.0,512.0,1024.0
4,4.0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96096.0,256GB SSD,None,256.0,256.0,512.0
5,5.0,Acer,Notebook,15.6,1366x768,AMD A9-Series 9420 3GHz,4,500GB HDD,AMD Radeon R5,Windows 10,2.10,21312.0,500GB HDD,None,500.0,500.0,1000.0
6,6.0,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.2GHz,16,256GB Flash Storage,Intel Iris Pro Graphics,Mac OS X,2.04,114018.0,256GB Flash Storage,None,256.0,256.0,512.0
7,7.0,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,256GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,61736.0,256GB Flash Storage,None,256.0,256.0,512.0
8,8.0,Asus,Ultrabook,14,Full HD 1920x1080,Intel Core i7 8550U 1.8GHz,16,512GB SSD,Nvidia GeForce MX150,Windows 10,1.30,79654.0,512GB SSD,None,512.0,512.0,1024.0
9,9.0,Acer,Ultrabook,14,IPS Panel Full HD 1920x1080,Intel Core i5 8250U 1.6GHz,8,256GB SSD,Intel UHD Graphics 620,Windows 10,1.60,41026.0,256GB SSD,None,256.0,256.0,512.0


In [15]:
# Exportamos la tabla limpia a un nuevo archivo CSV
# index=False evita que Python guarde la columna de los números de fila (0, 1, 2...)
df.to_csv('01_laptopData_Limpio_Python.csv', index=False)

In [16]:
# 1. Ahora SÍ extraemos los números del Disco_2
df['Cap_Disco_2_GB'] = df['Disco_2'].str.extract('(\d+)').astype(float)

# 2. Multiplicamos por 1000 los que dicen TB en el Disco_2
df.loc[df['Disco_2'].str.contains('TB', na=False), 'Cap_Disco_2_GB'] = df['Cap_Disco_2_GB'] * 1000

# 3. Volvemos a hacer la suma total con los datos corregidos
df['Capacidad_Total_GB'] = df['Cap_Disco_1_GB'].fillna(0) + df['Cap_Disco_2_GB'].fillna(0)

# 4. Comprobamos la corrección en los primeros registros
df[['Memory', 'Disco_1', 'Disco_2', 'Cap_Disco_1_GB', 'Cap_Disco_2_GB', 'Capacidad_Total_GB']].head(10)

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_10143/3010167015.py:2: SyntaxWarning: invalid escape sequence '\d'
  df['Cap_Disco_2_GB'] = df['Disco_2'].str.extract('(\d+)').astype(float)


,Memory,Disco_1,Disco_2,Cap_Disco_1_GB,Cap_Disco_2_GB,Capacidad_Total_GB
0,128GB SSD,128GB SSD,None,128.0,NaN,128.0
1,128GB Flash Storage,128GB Flash Storage,None,128.0,NaN,128.0
2,256GB SSD,256GB SSD,None,256.0,NaN,256.0
3,512GB SSD,512GB SSD,None,512.0,NaN,512.0
4,256GB SSD,256GB SSD,None,256.0,NaN,256.0
5,500GB HDD,500GB HDD,None,500.0,NaN,500.0
6,256GB Flash Storage,256GB Flash Storage,None,256.0,NaN,256.0
7,256GB Flash Storage,256GB Flash Storage,None,256.0,NaN,256.0
8,512GB SSD,512GB SSD,None,512.0,NaN,512.0
9,256GB SSD,256GB SSD,None,256.0,NaN,256.0


In [17]:
# Exportamos la tabla limpia a un nuevo archivo CSV
# index=False evita que Python guarde la columna de los números de fila (0, 1, 2...)
df.to_csv('01_laptopData_Limpio_Python.csv', index=False)